# Bonus — Product Category Translation
**Goal:** Translate Portuguese product category names to English, then identify which categories have the worst late delivery rates.

**Input:** `../data/02_delivered.parquet` + raw CSVs  
**Output:** `../data/05_delivered_cat.parquet`

---

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook_connected'
print('Libraries loaded.')

## Step 1 — Load data

In [ ]:
delivered = pd.read_parquet('../data/02_delivered.parquet')
products  = pd.read_csv('../data/olist_products_dataset.csv')
items     = pd.read_csv('../data/olist_order_items_dataset.csv')
transl    = pd.read_csv('../data/product_category_name_translation.csv', encoding='utf-8-sig')

print(f'delivered:     {len(delivered):,} orders')
print(f'products:      {len(products):,} products, {products["product_category_name"].nunique()} unique categories (Portuguese)')
print(f'translation:   {len(transl):,} category mappings')
print(f'\nSample Portuguese categories: {list(products["product_category_name"].dropna().unique()[:5])}')

## Step 2 — Translate category names
Join `products` to `translation` table to get English names.

In [ ]:
products_en = products.merge(
    transl[['product_category_name', 'product_category_name_english']],
    on='product_category_name',
    how='left'
)

translated = products_en['product_category_name_english'].notna().sum()
total_products = len(products_en)
print(f'Products with English category: {translated:,} / {total_products:,} ({translated/total_products*100:.1f}%)')

print('\nSample translations:')
print(products_en[['product_category_name', 'product_category_name_english']]
      .dropna().drop_duplicates().head(10).to_string(index=False))

## Step 3 — Attach category to each order
Each order can have multiple items. We take the **first item's category** per order to avoid row explosion.

In [ ]:
# One row per order — take first item
items_primary = (
    items[['order_id', 'product_id']]
    .drop_duplicates(subset='order_id', keep='first')
)
items_cat = items_primary.merge(
    products_en[['product_id', 'product_category_name_english']],
    on='product_id', how='left'
)

# Merge into delivered
delivered_cat = delivered.merge(items_cat, on='order_id', how='left')

# Clean up category names
delivered_cat['product_category_name_english'] = (
    delivered_cat['product_category_name_english']
    .fillna('Unknown')
    .str.replace('_', ' ')
    .str.title()
)

print(f'Orders with category attached: {(delivered_cat["product_category_name_english"] != "Unknown").sum():,}')
print(f'Unique English categories:     {delivered_cat["product_category_name_english"].nunique()}')
assert len(delivered_cat) == len(delivered), 'Row count changed — check join!'
print('✅ Row count unchanged.')

## Step 4 — Category-level late delivery rate

In [ ]:
cat_stats = (
    delivered_cat[delivered_cat['product_category_name_english'] != 'Unknown']
    .groupby('product_category_name_english')
    .agg(
        total      = ('order_id', 'count'),
        late       = ('delivery_status', lambda x: (x != 'On Time').sum()),
        avg_review = ('review_score', 'mean'),
    )
    .reset_index()
)
cat_stats['pct_late']   = (cat_stats['late'] / cat_stats['total'] * 100).round(1)
cat_stats['avg_review'] = cat_stats['avg_review'].round(2)

# Filter out low-volume categories
cat_stats = cat_stats[cat_stats['total'] >= 50]
print(f'Categories with ≥50 orders: {len(cat_stats)}')
print('\nTop 10 worst categories:')
print(cat_stats.nlargest(10, 'pct_late')[['product_category_name_english','total','pct_late','avg_review']].to_string(index=False))

## Step 5 — Visualise: top 20 categories by late rate

In [ ]:
top20 = cat_stats.nlargest(20, 'pct_late').sort_values('pct_late')

fig = px.bar(
    top20,
    x='pct_late', y='product_category_name_english',
    orientation='h',
    color='avg_review',
    color_continuous_scale='RdYlGn', range_color=[1, 5],
    text_auto='.1f',
    title='Top 20 Product Categories by Late Delivery Rate (min 50 orders)',
    labels={
        'pct_late': '% Late Deliveries',
        'product_category_name_english': 'Category (English)',
        'avg_review': 'Avg Review Score'
    },
    height=620,
)
fig.update_traces(texttemplate='%{x:.1f}%', textposition='outside')
fig.update_layout(coloraxis_colorbar_title='Avg Review')
fig.show()

## Step 6 — Save

In [ ]:
delivered_cat.to_parquet('../data/05_delivered_cat.parquet', index=False)
print('Saved → ../data/05_delivered_cat.parquet')
print('\nNext: run 06_pipeline_breakdown.ipynb')